In [50]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import DFDataset, TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [52]:
def add_gaussian_noise(df:pd.DataFrame, columns:List[str], mean:float = 0.0, std:float = 0.1):
    """
    Adds Gaussian noise to specified columns in a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to which noise will be added.
        columns (list): A list of column names in the DataFrame to which noise will be added.
        mean (int, optional): The mean of the Gaussian distribution. Defaults to 0.
        std (float, optional): The standard deviation of the Gaussian distribution. Defaults to 0.1.

    Returns:
        pandas.DataFrame: The DataFrame with added Gaussian noise to specified columns.
    """
    for col in columns:
        df[col] = df[col] + np.random.normal(loc=mean, scale=std, size=len(df))
    return df

In [53]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets

X = X.drop(columns=['Sex'], inplace=False)

df_data = pd.DataFrame(X, columns=abalone.data.feature_names)
df_data['target'] = y

scaler = MinMaxScaler()
df_data = pd.DataFrame(scaler.fit_transform(df_data), columns=df_data.columns)

X = df_data.drop(columns=['target'], inplace=False)
y = df_data['target']

In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pre-denoising XAI Benchmarck

In [55]:
model_params = {
    'ridge': {"alpha": 1.0},
    'pls': {"n_components": 1},
    'tree': {"max_depth": 5},
    'svm': {"dual": 'auto'},
    'knn': {"n_neighbors": 5, "weights": 'uniform', "algorithm": 'auto', "leaf_size": 30, "p": 2, "n_jobs": None},
}
xai_benchmark = XAI_benchmark(is_ts = False, model_params = model_params)

In [56]:
xai_benchmark.fit(X_train, y_train.values.ravel())

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [57]:
xai_benchmark.predict(X_test, y_test.values.ravel(), get_metrics=True)

({'ridge': array([ 0.38003037,  0.3275612 ,  0.43924189,  0.39214949,  0.34038061,
          0.33834242,  0.31818762,  0.27100008,  0.22968997,  0.34825369,
          0.38118863,  0.23758119,  0.37401718,  0.24148964,  0.27414924,
          0.40956867,  0.40843408,  0.33975017,  0.47395012,  0.26962057,
          0.64823593,  0.26758322,  0.36580726,  0.37264421,  0.30940389,
          0.25590614,  0.32409632,  0.26229435,  0.32014026,  0.30898338,
          0.33824979,  0.37167707,  0.22961759,  0.31457676,  0.29328786,
          0.29407342,  0.20137304,  0.12366368,  0.25910555,  0.21811942,
          0.39917119,  0.5677922 ,  0.40025956,  0.30591948,  0.35541355,
          0.38227802,  0.30652937,  0.29525298,  0.38921719,  0.15909516,
          0.21778531,  0.60732457,  0.32468314,  0.14615722,  0.21936232,
          0.34905272,  0.35931662,  0.45979775,  0.3367447 ,  0.22910169,
          0.38059801,  0.44458516,  0.3436406 ,  0.24923187,  0.25361468,
          0.35727369,  0.1623

# Denoise the entire noisy dataframe

In [58]:
df_denoised = pd.DataFrame()
for col in X.columns:
    df_denoised[col] = X[col].rolling(window=5, min_periods=1).mean()
df_denoised['y'] = y.rolling(window=5, min_periods=1).mean()

In [59]:
X_denoised = df_denoised.drop(columns=['y'], inplace=False)
y_denoised = df_denoised['y']

In [60]:
X_train_denoised, X_test_denoised, y_train_denoised, y_test_denoised = train_test_split(
    X_denoised.values, y_denoised.values, test_size=0.2, random_state=42
)

In [61]:
xai_benchmark.fit(X_train_denoised, y_train_denoised)
xai_benchmark.predict(X_test_denoised, y_test_denoised, get_metrics=True)

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


({'ridge': array([0.34467444, 0.29320179, 0.43381056, 0.34345941, 0.36216946,
         0.29718425, 0.29123297, 0.24380222, 0.24487656, 0.31359727,
         0.29387837, 0.28095334, 0.27644442, 0.27481208, 0.29577436,
         0.41759208, 0.25320883, 0.31751564, 0.46450056, 0.2584384 ,
         0.65359944, 0.30651658, 0.37334245, 0.35498553, 0.31306172,
         0.29161095, 0.27892006, 0.3610014 , 0.38059845, 0.30034197,
         0.30332158, 0.32054334, 0.21223468, 0.36383787, 0.29576688,
         0.28648656, 0.25386128, 0.1798098 , 0.40154764, 0.21213676,
         0.33111477, 0.41617709, 0.40217114, 0.27903374, 0.28912695,
         0.3217904 , 0.28545886, 0.27297761, 0.37307081, 0.22509567,
         0.23749194, 0.38703751, 0.32081051, 0.21917716, 0.21903491,
         0.31615578, 0.45584292, 0.28169759, 0.34090041, 0.28959831,
         0.32801214, 0.33987636, 0.33345171, 0.23343245, 0.43926298,
         0.35749427, 0.16301689, 0.28933545, 0.26835385, 0.45846353,
         0.35348824, 0.33

In [62]:
xai_benchmark.predict(X_test.values, y_test.values.ravel(), get_metrics=True)

({'ridge': array([ 0.40207057,  0.32698842,  0.47232546,  0.39281176,  0.34990408,
          0.3516838 ,  0.32152821,  0.25264927,  0.2326916 ,  0.34603755,
          0.38374085,  0.24820868,  0.40275794,  0.24830383,  0.26783998,
          0.42100779,  0.43133372,  0.34293173,  0.4898857 ,  0.26257923,
          0.72617843,  0.23706582,  0.35340384,  0.37588987,  0.29316911,
          0.25667184,  0.32968827,  0.24778592,  0.31512841,  0.30288582,
          0.34591119,  0.38124992,  0.2332324 ,  0.32067631,  0.29221663,
          0.26366995,  0.20909315,  0.13983464,  0.25754477,  0.21948078,
          0.39702763,  0.62041828,  0.41283931,  0.30342665,  0.36348409,
          0.35020478,  0.28306628,  0.27583993,  0.4109637 ,  0.17248055,
          0.22308257,  0.64173547,  0.32778556,  0.15845379,  0.21626343,
          0.34743037,  0.35716473,  0.47047122,  0.33361057,  0.2368141 ,
          0.36304221,  0.46461889,  0.35155549,  0.25484332,  0.25063306,
          0.33687746,  0.1713